<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/SQL/SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# วิเคราห์ข้อมูลด้วย SQL

---

IMPORT ฟังก์ชันสำเร็จรูป



In [ ]:
import sqlite3
import pandas as pd

# โหลดไฟล์ csv. เพื่อนำมาใช้ในการวิเคราะห์

In [ ]:
# 1. โหลดไฟล์ CSV ทั้ง 3 ไฟล์
df_customers = pd.read_csv('https://raw.githubusercontent.com/natchanant-arch/Project_Savings_Cooperative/refs/heads/First/%E0%B8%A5%E0%B8%B9%E0%B8%81%E0%B8%84%E0%B9%89%E0%B8%B2%E0%B8%AA%E0%B8%AB%E0%B8%81%E0%B8%A3%E0%B8%93%E0%B9%8C%E0%B8%AD%E0%B8%AD%E0%B8%A1%E0%B8%97%E0%B8%A3%E0%B8%B1%E0%B8%9E%E0%B8%A2%E0%B9%8C.csv')
df_transactions = pd.read_csv('https://raw.githubusercontent.com/natchanant-arch/Project_Savings_Cooperative/refs/heads/First/%E0%B8%98%E0%B8%B8%E0%B8%A3%E0%B8%81%E0%B8%A3%E0%B8%A3%E0%B8%A1%E0%B8%AA%E0%B8%AB%E0%B8%81%E0%B8%A3%E0%B8%93%E0%B9%8C%E0%B8%AD%E0%B8%AD%E0%B8%A1%E0%B8%97%E0%B8%A3%E0%B8%B1%E0%B8%9E%E0%B8%A2%E0%B9%8C%20(1).csv')
df_all = pd.read_csv('https://raw.githubusercontent.com/natchanant-arch/Project_Savings_Cooperative/refs/heads/First/%E0%B8%AA%E0%B8%AB%E0%B8%81%E0%B8%A3%E0%B8%93%E0%B9%8C%E0%B8%AD%E0%B8%AD%E0%B8%A1%E0%B8%97%E0%B8%A3%E0%B8%B1%E0%B8%9E%E0%B8%A2%E0%B9%8C.csv')

# (ตัวเลือกเสริม) ลบ columna 'Unnamed: 0' ที่มักจะติดมาจากการเซฟไฟล์ CSV ออกไปเพื่อความสะอาด
for df in [df_customers, df_transactions, df_all]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)

# 2. เชื่อมต่อกับ SQLite ฐานข้อมูลในหน่วยความจำ (In-memory database)
conn = sqlite3.connect('coop.db')

# 3. บันทึก Pandas DataFrame ลงไปเป็นตารางในฐานข้อมูล SQLite
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_transactions.to_sql('transactions', conn, if_exists='replace', index=False)
df_all.to_sql('all_data', conn, if_exists='replace', index=False)

print("นำเข้าข้อมูลเข้า SQLite สำเร็จเรียบร้อยแล้ว!")

# วิเคราะห์ข้อมูลจากการใช้ตาราง csv. 1 ตาราง

---



# ใครคือลูกค้าที่มียอดธุรกรรมน้อยสุด

In [ ]:
query_1 = """
    SELECT customer_name, transaction_type, amount, status
    FROM all_data
    WHERE status = "สำเร็จ"
    ORDER BY amount ASC
"""
df = pd.read_sql(query_1, con=conn)
df.head()

# ใครคือลูกค้าที่มียอดธุรกรรมมากสุด

In [ ]:
query_2 = """
    SELECT customer_name, transaction_type, amount, status
    FROM all_data
    WHERE status = "สำเร็จ"
    ORDER BY amount DESC
"""
df = pd.read_sql(query_2, con=conn)
df.head()

# จำนวนธุรกรรมแต่ละประเภท มีผู้ใช้บริการไปทั้งหมดกี่ครั้ง

In [ ]:
query_3 = """
    SELECT transaction_type, COUNT(*) AS transaction_count
    FROM all_data
    GROUP BY transaction_type
    ORDER BY transaction_count DESC;
"""
df = pd.read_sql(query_3, con=conn)
df.head()

# วิเคราะห์ข้อมูลจาก SQL กรณีที่มีการ Join 2 ตารางเข้าด้วยกัน

In [ ]:
import pandas as pd
from google.cloud import client

# ใส่ Project ID ของคุณ
project_id = "academic-works-506403-n9"

# สรุปยอดรวมแต่ละธุรกรรม

In [ ]:
query ="""
SELECT
  COUNT(t.txn_id) AS Total_txn,
  s.transaction_type,
  SUM(s.amount) AS Total_amount,
  AVG(s.amount) AS Total_avg,
  MAX(s.amount) AS Maxim
FROM `my-type-project-505203.11.ลูกค้า` AS t
JOIN `my-type-project-505203.11.ธุรกรรม` AS s
  ON t.int64_field_0 = s.int64_field_0
Where s.status = 'สำเร็จ'
GROUP BY s.transaction_type
ORDER BY s.transaction_type  """
df = pd.read_gbq(query, project_id=project_id, dialect="standard")
df.head()

#ลูกค้าคนใด ทำธุรกรรมน้อยสุด 5 อันดับแรก

In [ ]:
query ="""
SELECT
  t.customer_name,
  s.transaction_type,
  s.amount
FROM `my-type-project-505203.11.ลูกค้า` AS t
JOIN `my-type-project-505203.11.ธุรกรรม` AS s
  ON t.int64_field_0 = s.int64_field_0
Where   s.status = 'สำเร็จ'
ORDER BY s.amount asc
LIMIT 5"""
df = pd.read_gbq(query, project_id=project_id, dialect="standard")
df.head()

#ลูกค้าคนใด ฝากเงินเยอะสุด 5 อันดับแรก

In [ ]:
query ="""
SELECT
  t.customer_name,
  s.transaction_type,
  s.amount
FROM `my-type-project-505203.11.ลูกค้า` AS t
JOIN `my-type-project-505203.11.ธุรกรรม` AS s
  ON t.int64_field_0 = s.int64_field_0
WHERE s.transaction_type = 'ฝากเงิน'
ORDER BY s.amount DESC
LIMIT 5"""
df = pd.read_gbq(query, project_id=project_id, dialect="standard")
df.head()

#ยอดเงินในบัญชีสูงสุด 5 อันดับแรก

In [ ]:
query ="""
SELECT
  t.customer_name,
  s.balance_after
FROM `my-type-project-505203.11.ลูกค้า` AS t
JOIN `my-type-project-505203.11.ธุรกรรม` AS s
  ON t.int64_field_0 = s.int64_field_0
ORDER BY s.balance_after DESC
LIMIT 5"""
df = pd.read_gbq(query, project_id=project_id, dialect="standard")
df.head()